In [1]:
import hashlib, yaml, boto3, pickle
from pathlib import Path
from datetime import datetime

# (1) run_manifest.yml 로드

In [2]:
with open('run_manifest.yml') as f:
    manifest = yaml.safe_load(f)

print(f"재현 대상 run_id: {manifest['run_id']}")
print(f"원본 실행 시각: {manifest['created_at']}")

재현 대상 run_id: 20260305_lightgbm_baseline_b307c3ad
원본 실행 시각: 2026-03-05T11:49:49.649865


# (2) config_hashes 검증

In [20]:
def get_file_hash(filepath: Path) -> str:
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            sha256.update(chunk)
    return f"sha256:{sha256.hexdigest()[:16]}..."

# S3 conf 버킷에서 설정 파일 다운로드
s3 = boto3.client('s3')
conf_s3_path = manifest['s3_refs']['conf_path']
# 예: s3://gs-retail-awesome-conf-us-east-1/dev/sean/.../baseline-v1/
bucket = conf_s3_path.split('/')[2]
prefix = '/'.join(conf_s3_path.split('/')[3:])

CONF_DIR = Path("conf")
CONF_DIR.mkdir(exist_ok=True)

for fname in ['env.yml', 'meta.yml', 'model.yml']:
    s3.download_file(bucket, prefix + fname, str(CONF_DIR / fname))

# 해시 비교
expected = manifest['config_hashes']
for key, fname in [('env_yml','env.yml'), ('meta_yml','meta.yml'), ('model_yml','model.yml')]:
    computed = get_file_hash(CONF_DIR / fname)
    match = "✅ OK" if computed == expected[key] else "❌ MISMATCH"
    print(f"{fname}: {match}  (expected={expected[key]}, got={computed})")

# 불일치 시 중단
assert all(
    get_file_hash(CONF_DIR / f) == expected[k]
    for k, f in [('env_yml','env.yml'),('meta_yml','meta.yml'),('model_yml','model.yml')]
), "Config hash 불일치 — 설정이 변경되었습니다. 재현 불가."

env.yml: ✅ OK  (expected=sha256:bce9035b4fa7fe32..., got=sha256:bce9035b4fa7fe32...)
meta.yml: ✅ OK  (expected=sha256:ab1292a82f079a61..., got=sha256:ab1292a82f079a61...)
model.yml: ✅ OK  (expected=sha256:fbec613de1c2fa0e..., got=sha256:fbec613de1c2fa0e...)


# (3) 동일한 설정으로 재실행

## 검증된 설정 파일(conf) 로드

In [21]:
def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

env_config   = load_yaml(CONF_DIR / 'env.yml')
meta_config  = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')

print(env_config)

{'env': 'dev', 'region': 'us-east-1', 's3': {'conf_bucket': 'gs-retail-awesome-conf-us-east-1', 'data_bucket': 'gs-retail-awesome-data-us-east-1', 'model_bucket': 'gs-retail-awesome-model-us-east-1'}, 's3_paths': {'conf': '{env}/{user_id}/{project}/{experiment}/', 'data': '{env}/{user_id}/{project}/{version}/', 'model': '{env}/{user_id}/{project}/{experiment}/{run_id}/'}, 'local': {'root': 'run_pm', 'input_dir': 'input', 'output_dir': 'output', 'output_subdirs': ['metadata', 'config', 'data_refs', 'artifacts/model', 'artifacts/metrics', 'artifacts/charts', 'artifacts/explainability', 'reports']}}


## 이전 학습 시 사용된 데이터, S3버킷에서 다운로드하기

In [24]:
data_s3_path = manifest['s3_refs']['data_path']
data_bucket  = data_s3_path.split('/')[2]
data_prefix  = '/'.join(data_s3_path.split('/')[3:])

print(data_bucket, data_prefix,data_s3_path)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

for fname in ['train.csv', 'test.csv', 'validation.csv']:
    s3.download_file(data_bucket, data_prefix +'data/'+ fname, str(DATA_DIR / fname))

gs-retail-awesome-data-us-east-1 dev/sean/titanic-survival-prediction/v1.0/ s3://gs-retail-awesome-data-us-east-1/dev/sean/titanic-survival-prediction/v1.0/
